In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab data access)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# 5-Class Emergency Acuity Triage: Final 2-Stage Stacking Pipeline (LightGBM + Random Forest with Class Weights) & Logistic Regressor on Reduced 20 Features (`models/train_oof_logistic_regression_stacking_new.ipynb`)

This notebook implements the **2-Stage Hierarchical Stacking Pipeline** trained exclusively on the **Reduced 20 Arrival Features** (min/max stay vitals and stay ranges completely excluded) for all **5 triage levels (`ESI 1..5`)** on the **5-variable Emergency Department dataset (`datasets/5v_cleandf.RData`)**:

### 🏗️ Architecture & Model Specification
1. **Feature Matrix: Reduced 20 Arrival Features**:
   - Pure arrival vitals and demographics (`age`, `gender`, `cc_breathingdifficulty`, `triage_vital_hr`, `triage_vital_sbp`, `triage_vital_rr`, `triage_vital_o2`).
   - 10 Binary clinical anomaly flags (dyspnea, bradypnea, tachypnea, hypotension, hypertension, bradycardia, tachycardia).
   - 3 Pure arrival composite indices: `shock_index` ($HR / SBP$), `rox_index` ($O_2 / RR$), `bif` ($(RR / O_2) \times 100$).
   - *No stay duration min/max vitals or range dynamics are included.*
2. **Layer 1: Resuscitation Detection Sub-Model (ESI 1 vs Non-ESI 1)**:
   - Binary classifier identifying critical **`ESI 1`** resuscitation cases from stable **`ESI 2..5`**.
   - Model: **`lightgbm.LGBMClassifier`** trained with binary **SMOTE** oversampling on the training partition.
   - Produces resuscitation posterior probability $p_1 = P(\text{ESI 1})$.
3. **Layer 2: Non-ESI 1 Multi-Class Sub-Model (ESI 2, 3, 4, 5)**:
   - Standard **4-Class Random Forest Classifier (`RandomForestClassifier`)** trained with **`class_weight='balanced'`** on the non-ESI 1 cohort ($y \ne 1$).
   - Produces conditional posterior probabilities $[q_2, q_3, q_4, q_5]$ for classes `2, 3, 4, 5` such that $\sum_{k=2}^5 q_k = 1.0$.
4. **Hierarchical Probability Fusion**:
   $$\begin{aligned}
   P(\text{ESI 1}) &= p_1 \\
   P(\text{ESI 2}) &= (1 - p_1) \cdot q_2 \\
   P(\text{ESI 3}) &= (1 - p_1) \cdot q_3 \\
   P(\text{ESI 4}) &= (1 - p_1) \cdot q_4 \\
   P(\text{ESI 5}) &= (1 - p_1) \cdot q_5
   \end{aligned}$$
   $$\sum_{k=1}^5 P_k = p_1 + (1 - p_1) \cdot 1.0 = 1.0$$
5. **Logistic Regressor Meta-Learner Calibration**:
   - Multinomial Logistic Regression (`LogisticRegression(class_weight='balanced')`) calibrated on the validation probability simplex $[P_1, P_2, P_3, P_4, P_5]$.

```mermaid
flowchart TD
    Data["Complete Dataset (5v_cleandf.RData)"] --> Split["Stratified 3-Way Split: Train (98%), Val (1%), Test (1%)"]
    Split --> Preproc["Reduced 20 Arrival Features & Standardization"]
    
    Preproc --> L1["Layer 1: Binary LightGBM (SMOTE: ESI 1 vs 2..5)"]
    Preproc --> L2["Layer 2: 4-Class Random Forest (class_weight='balanced': ESI 2, 3, 4, 5 on Non-ESI 1)"]
    
    L1 & L2 --> HierFusion["Hierarchical Probability Fusion:<br/>P1 = p1<br/>P_k = (1 - p1) * q_k"]
    HierFusion --> Meta["Multinomial Logistic Regression Meta-Learner"]
    
    Meta --> Benchmark["Holdout Test Benchmark: Per-Class Scoring Breakdown, 5x5 Confusion Matrix, Multiclass ROC Curves"]
```

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Raw Dataset, Filter Complete Cases & Stratified 3-Way Split (Train/Val/Test)
# ---------------------------------------------------------
suppressPackageStartupMessages({
  library(jsonlite)
  library(dplyr)
})

config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) config_path <- "config/triage_conf.json"
config <- fromJSON(config_path)

set.seed(config$training$random_state)

stratified_sample <- function(y, fraction, seed = 42) {
  set.seed(seed)
  idx_list <- split(seq_along(y), y)
  sampled <- unlist(lapply(idx_list, function(idx) {
    n_sample <- max(1, round(length(idx) * fraction))
    sample(idx, size = n_sample)
  }))
  return(sort(sampled))
}

data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) data_file <- paste0("../", data_file)

data_env <- new.env()
load(data_file, envir = data_env)

df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
raw_df   <- get(df_names[which.max(df_sizes)], envir = data_env)
target_col_name <- config$classes$target_col

initial_total_rows <- nrow(raw_df)
cat("========================================================================\n")
cat(sprintf("  INITIAL DATASET LOADED: %d Total Rows, %d Total Columns\n", initial_total_rows, ncol(raw_df)))
cat("========================================================================\n")
if (target_col_name %in% names(raw_df)) {
  cat("Initial ESI Target Distribution (including NAs):\n")
  print(table(raw_df[[target_col_name]], useNA = "ifany"))
  cat("------------------------------------------------------------------------\n")
}

gender_vec <- if ("gender" %in% names(raw_df)) ifelse(is.na(raw_df$gender), NA, ifelse(as.character(raw_df$gender) == "Male", 1, 0)) else rep(NA, nrow(raw_df))
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) raw_df$cc_breathingdifficulty else rep(NA, nrow(raw_df))

get_vec <- function(col_name) {
  if (col_name %in% names(raw_df)) {
    return(raw_df[[col_name]])
  } else {
    return(rep(NA, nrow(raw_df)))
  }
}

raw_esi <- as.character(raw_df[[target_col_name]])

# Load arrival features only
df_master <- data.frame(
  age                     = raw_df$age,
  cc_breathingdifficulty  = cc_bd_vec,
  gender                  = gender_vec,
  triage_vital_hr         = get_vec("triage_vital_hr"),
  triage_vital_sbp        = get_vec("triage_vital_sbp"),
  triage_vital_rr         = get_vec("triage_vital_rr"),
  triage_vital_o2         = get_vec("triage_vital_o2"),
  target_col              = factor(raw_esi, levels = c("1", "2", "3", "4", "5"))
)

# Strictly drop any row containing at least 1 null/NA value across arrival features or target
df_master <- na.omit(df_master)
df_master$target_num <- as.numeric(as.character(df_master$target_col))

clean_total_rows <- nrow(df_master)
dropped_rows     <- initial_total_rows - clean_total_rows

cat(sprintf("Missing Values Filter: Dropped %d rows with >= 1 NA feature (Retained %d Complete Cases, %.2f%%)\n", 
            dropped_rows, clean_total_rows, (clean_total_rows / initial_total_rows) * 100))
cat("Cleaned ESI Distribution (100% complete cases):\n")
print(table(df_master$target_col))
cat("------------------------------------------------------------------------\n")

# Stratified 3-Way Partitioning based on triage_conf.json
test_size <- config$training$test_size
val_size  <- config$training$val_size
seed_val  <- config$training$random_state

# 1. Extract Stratified Holdout Test Set (e.g., 1%)
idx_test <- stratified_sample(df_master$target_col, test_size, seed = seed_val)
test_df_clean  <- df_master[idx_test, ]
rem_df         <- df_master[-idx_test, ]

# 2. Extract Stratified Validation Set from remainder (e.g., 1% of total)
val_adj_fraction <- val_size / (1 - test_size)
idx_val <- stratified_sample(rem_df$target_col, val_adj_fraction, seed = seed_val + 1)
val_df_clean   <- rem_df[idx_val, ]
train_df_clean <- rem_df[-idx_val, ]

train_mat_export <- as.matrix(cbind(train_df_clean[, 1:7], target = train_df_clean$target_num))
val_mat_export   <- as.matrix(cbind(val_df_clean[, 1:7],   target = val_df_clean$target_num))
test_mat_export  <- as.matrix(cbind(test_df_clean[, 1:7],  target = test_df_clean$target_num))

cat(sprintf("3-Way Partition Complete:\n  Train Set      = %d rows (%.2f%%)\n  Validation Set = %d rows (%.2f%%)\n  Holdout Test   = %d rows (%.2f%%)\n", 
            nrow(train_mat_export), (nrow(train_mat_export) / clean_total_rows) * 100,
            nrow(val_mat_export),   (nrow(val_mat_export) / clean_total_rows) * 100,
            nrow(test_mat_export),  (nrow(test_mat_export) / clean_total_rows) * 100))
cat("========================================================================\n")

In [ ]:
# ---------------------------------------------------------
# Step 2: Feature Matrix Construction & Standardization (Reduced 20 Features Only)
# ---------------------------------------------------------
import os, json, pickle, time, warnings
import numpy as np, pandas as pd
from rpy2.robjects import r
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, recall_score,
    precision_score, f1_score, roc_auc_score, average_precision_score, confusion_matrix
)
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
ROOT = '..' if os.path.basename(os.getcwd()) == 'models' else '.'

train_mat_in = np.array(r('train_mat_export'), dtype=np.float64)
val_mat_in   = np.array(r('val_mat_export'),   dtype=np.float64)
test_mat_in  = np.array(r('test_mat_export'),  dtype=np.float64)

raw_mat_tr  = train_mat_in[:, :7]
y_train     = train_mat_in[:, 7].astype(int)

raw_mat_val = val_mat_in[:, :7]
y_val       = val_mat_in[:, 7].astype(int)

raw_mat_ts  = test_mat_in[:, :7]
y_test      = test_mat_in[:, 7].astype(int)

def build_20_feature_matrix(raw_mat):
    N = len(raw_mat)
    X = np.zeros((N, 20), dtype=np.float64)
    # 7 Raw Arrival Features (Indices 0..6: age, cc_bd, gender, triage_hr, triage_sbp, triage_rr, triage_o2)
    X[:, :7] = raw_mat[:, :7]
    
    t_hr = raw_mat[:, 3]; t_sbp = raw_mat[:, 4]; t_rr = raw_mat[:, 5]; t_o2 = raw_mat[:, 6]
    
    # 10 Binary Arrival Vital Flags (Indices 7..16)
    X[:, 7]  = (t_o2 < 90).astype(float)                          # is_dyspnea_total
    X[:, 8]  = ((t_o2 >= 90) & (t_o2 < 94)).astype(float)         # is_dyspnea_moderate
    X[:, 9]  = (t_rr < 10).astype(float)                          # is_bradypnea
    X[:, 10] = (t_rr > 30).astype(float)                          # is_tachypnea
    X[:, 11] = (t_sbp <= 90).astype(float)                        # is_hypotension
    X[:, 12] = (t_sbp > 220).astype(float)                        # is_hypertension
    X[:, 13] = (t_hr < 40).astype(float)                          # is_bradycardia_total
    X[:, 14] = ((t_hr >= 40) & (t_hr < 60)).astype(float)         # is_bradycardia_moderate
    X[:, 15] = (t_hr > 150).astype(float)                         # is_tachycardia_total
    X[:, 16] = ((t_hr > 100) & (t_hr <= 150)).astype(float)       # is_tachycardia_moderate
    
    # 3 Pure Arrival Clinical Composite Indices (Indices 17..19)
    X[:, 17] = t_hr / np.where(t_sbp == 0, 1.0, t_sbp)            # shock_index
    X[:, 18] = t_o2 / np.where(t_rr == 0, 1.0, t_rr)              # rox_index
    X[:, 19] = (t_rr / np.where(t_o2 == 0, 1.0, t_o2)) * 100.0   # bif
    return X

X_train_raw = build_20_feature_matrix(raw_mat_tr)
X_val_raw   = build_20_feature_matrix(raw_mat_val)
X_test_raw  = build_20_feature_matrix(raw_mat_ts)

cont_cols = [0, 3, 4, 5, 6, 17, 18, 19]
scaler = StandardScaler()
X_train = X_train_raw.copy()
X_val   = X_val_raw.copy()
X_test  = X_test_raw.copy()
X_train[:, cont_cols] = scaler.fit_transform(X_train_raw[:, cont_cols])
X_val[:, cont_cols]   = scaler.transform(X_val_raw[:, cont_cols])
X_test[:, cont_cols]  = scaler.transform(X_test_raw[:, cont_cols])

feature_names_20 = [
    'age', 'cc_breathingdifficulty', 'gender', 'triage_vital_hr', 'triage_vital_sbp', 'triage_vital_rr', 'triage_vital_o2',
    'is_dyspnea_total', 'is_dyspnea_moderate', 'is_bradypnea', 'is_tachypnea', 'is_hypotension', 'is_hypertension',
    'is_bradycardia_total', 'is_bradycardia_moderate', 'is_tachycardia_total', 'is_tachycardia_moderate',
    'shock_index', 'rox_index', 'bif'
]

print(f"Reduced 20-Feature Matrices Ready: Train={X_train.shape}, Val={X_val.shape}, Test={X_test.shape}")

In [ ]:
# ---------------------------------------------------------
# Step 3: Train Layer 1 Sub-Model (ESI 1 vs Non-ESI 1 with LightGBM + SMOTE)
# ---------------------------------------------------------
print("=" * 80)
print("  TRAINING LAYER 1: RESUSCITATION DETECTION (LIGHTGBM + SMOTE ON 20 FEATURES)")
print("=" * 80)

def binary_numpy_smote(X, y_bin, seed=42):
    np.random.seed(seed)
    pos_mask = (y_bin == 1)
    neg_mask = (y_bin == 0)
    n_pos = np.sum(pos_mask)
    n_neg = np.sum(neg_mask)
    if n_pos == 0 or n_neg == 0 or n_pos == n_neg:
        return X, y_bin
    if n_pos < n_neg:
        min_X = X[pos_mask]; target_syn = n_neg - n_pos; min_label = 1
    else:
        min_X = X[neg_mask]; target_syn = n_pos - n_neg; min_label = 0
    n_min = len(min_X)
    syn_X = np.zeros((target_syn, X.shape[1]), dtype=np.float64)
    for i in range(target_syn):
        idx1 = np.random.randint(0, n_min)
        idx2 = np.random.randint(0, n_min)
        alpha = np.random.rand()
        syn_X[i] = min_X[idx1] + alpha * (min_X[idx2] - min_X[idx1])
    return np.vstack([X, syn_X]), np.hstack([y_bin, np.full(target_syn, min_label)])

lgb_params = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'max_depth': 6,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 1,
    'verbosity': -1,
    'random_state': 42
}

t0 = time.time()

# Layer 1: ESI 1 vs (ESI 2..5) trained with SMOTE
X_sm1, y_sm1 = binary_numpy_smote(X_train, (y_train == 1).astype(int))
l1_model = lgb.LGBMClassifier(**lgb_params, n_estimators=100)
l1_model.fit(X_sm1, y_sm1, eval_set=[(X_val, (y_val == 1).astype(int))], callbacks=[lgb.early_stopping(10, verbose=False)])

print(f"✓ Layer 1 (ESI 1 LightGBM) trained in {time.time()-t0:.1f}s!")

In [ ]:
# ---------------------------------------------------------
# Step 4: Train Layer 2 Sub-Model (4-Class Random Forest with class_weight='balanced')
# ---------------------------------------------------------
print("=" * 80)
print("  TRAINING LAYER 2: 4-CLASS RANDOM FOREST (ESI 2, 3, 4, 5 WITH class_weight='balanced')")
print("=" * 80)

rf_params = {
    'n_estimators': 150,
    'max_depth': 12,
    'min_samples_split': 10,
    'min_samples_leaf': 4,
    'class_weight': 'balanced',
    'random_state': 42,
    'n_jobs': -1
}

t0 = time.time()

# Filter non-ESI 1 cases for Layer 2
m2_tr  = (y_train != 1)
m2_val = (y_val != 1)

# Train 4-Class Random Forest with class_weight='balanced'
l2_rf_model = RandomForestClassifier(**rf_params)
l2_rf_model.fit(X_train[m2_tr], y_train[m2_tr])

print(f"✓ Layer 2 (4-Class Random Forest with class_weight='balanced') trained in {time.time()-t0:.1f}s!")

In [ ]:
# ---------------------------------------------------------
# Step 5: Hierarchical Probability Fusion & Multinomial Logistic Regression Meta-Learner
# ---------------------------------------------------------
print("=" * 80)
print("  HIERARCHICAL PROBABILITY FUSION & LOGISTIC REGRESSOR CALIBRATION")
print("=" * 80)

def compute_2stage_probs(l1_binary, l2_rf, X_in):
    p1 = l1_binary.predict_proba(X_in)[:, 1]   # P(ESI 1)
    q_2345 = l2_rf.predict_proba(X_in)         # Shape (N, 4) corresponding to classes [2, 3, 4, 5]
    
    P = np.zeros((len(X_in), 5))
    P[:, 0] = p1
    P[:, 1] = (1 - p1) * q_2345[:, 0]  # ESI 2
    P[:, 2] = (1 - p1) * q_2345[:, 1]  # ESI 3
    P[:, 3] = (1 - p1) * q_2345[:, 2]  # ESI 4
    P[:, 4] = (1 - p1) * q_2345[:, 3]  # ESI 5
    return P

val_probs  = compute_2stage_probs(l1_model, l2_rf_model, X_val)
test_probs = compute_2stage_probs(l1_model, l2_rf_model, X_test)

# Fit Multinomial Logistic Regression Meta-Learner on Validation Hierarchical Probabilities
meta_learner = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
meta_learner.fit(val_probs, y_val)

preds_test = meta_learner.predict(test_probs)
probs_test = meta_learner.predict_proba(test_probs)

print("✓ Meta-Learner Logistic Regression successfully fitted and calibrated on validation probability simplex!")

In [ ]:
# ---------------------------------------------------------
# Step 6: Holdout Test Set Benchmark & Comprehensive Evaluation
# ---------------------------------------------------------
def get_per_class_breakdown(y_true, y_pred, probs, pipeline_name, class_list=[1, 2, 3, 4, 5]):
    rows = []
    recalls, specs, bal_accs, aucs = [], [], [], []
    for idx, cls in enumerate(class_list):
        y_bin_true = (y_true == cls).astype(int)
        y_bin_pred = (y_pred == cls).astype(int)
        tp = np.sum((y_bin_true == 1) & (y_bin_pred == 1))
        fn = np.sum((y_bin_true == 1) & (y_bin_pred == 0))
        fp = np.sum((y_bin_true == 0) & (y_bin_pred == 1))
        tn = np.sum((y_bin_true == 0) & (y_bin_pred == 0))
        rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        bal  = (rec + spec) / 2.0
        try:
            auc = roc_auc_score(y_bin_true, probs[:, idx])
        except Exception:
            auc = 0.0
        recalls.append(rec); specs.append(spec); bal_accs.append(bal); aucs.append(auc)
        rows.append({
            'Pipeline': pipeline_name,
            'Class': f'ESI_{cls}',
            'Recall': round(rec, 4),
            'Specificity': round(spec, 4),
            'Balanced_Accuracy': round(bal, 4),
            'ROC_AUC': round(auc, 4)
        })
    rows.append({
        'Pipeline': pipeline_name,
        'Class': 'Macro_Average',
        'Recall': round(np.mean(recalls), 4),
        'Specificity': round(np.mean(specs), 4),
        'Balanced_Accuracy': round(np.mean(bal_accs), 4),
        'ROC_AUC': round(np.mean(aucs), 4)
    })
    return pd.DataFrame(rows)

def evaluate_binary_submodel(y_true, y_pred, y_prob, model_name, pos_label_name="Positive"):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    bal_acc = (sens + spec) / 2.0
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    f1 = 2 * (prec * sens) / (prec + sens) if (prec + sens) > 0 else 0.0
    try: auc_val = roc_auc_score(y_true, y_prob)
    except Exception: auc_val = 0.0
    try: pr_auc = average_precision_score(y_true, y_prob)
    except Exception: pr_auc = 0.0
    
    return {
        'Model': model_name,
        f'Recall_Sensitivity_{pos_label_name}': round(sens, 4),
        'Specificity_Negative': round(spec, 4),
        'Balanced_Accuracy': round(bal_acc, 4),
        'Precision_PPV': round(prec, 4),
        'F1_Score': round(f1, 4),
        'ROC_AUC': round(auc_val, 4),
        'PR_AUC': round(pr_auc, 4),
        'TP': int(tp), 'FP': int(fp), 'TN': int(tn), 'FN': int(fn)
    }

# ---------------------------------------------------------
# Part A: STANDALONE LAYER 1 BINARY EVALUATION (ESI 1 vs Non-ESI 1)
# ---------------------------------------------------------
y_test_l1 = (y_test == 1).astype(int)
prob_l1 = l1_model.predict_proba(X_test)[:, 1]
pred_l1 = (prob_l1 >= 0.50).astype(int)

l1_res = [evaluate_binary_submodel(y_test_l1, pred_l1, prob_l1, 'Layer 1 (LightGBM + SMOTE on 20 Features, p>=0.50)', pos_label_name='ESI1')]
l1_report_df = pd.DataFrame(l1_res)

print("=" * 110)
print("   PART A: STANDALONE LAYER 1 BINARY EVALUATION (ESI 1 vs Non-ESI 1, WITHOUT LOGREG)")
print("=" * 110)
print(l1_report_df[['Model', 'Recall_Sensitivity_ESI1', 'Specificity_Negative', 'Balanced_Accuracy', 'Precision_PPV', 'F1_Score', 'ROC_AUC', 'PR_AUC']].to_string(index=False))
print("-" * 110)
print(f"Layer 1 Confusion Breakdown: TP={l1_res[0]['TP']}, FP={l1_res[0]['FP']}, TN={l1_res[0]['TN']}, FN={l1_res[0]['FN']}")
print("=" * 110 + chr(10))

# ---------------------------------------------------------
# Part B: STANDALONE LAYER 2 4-CLASS RANDOM FOREST EVALUATION (ESI 2, 3, 4, 5 on Non-ESI 1)
# ---------------------------------------------------------
m2_ts = (y_test != 1)
y_test_non1 = y_test[m2_ts]

probs_rf = l2_rf_model.predict_proba(X_test[m2_ts])
preds_rf = l2_rf_model.predict(X_test[m2_ts])

report_l2_rf = get_per_class_breakdown(y_test_non1, preds_rf, probs_rf, 'Layer 2 4-Class RF (class_weight=balanced on 20 Features)', class_list=[2, 3, 4, 5])

print("=" * 105)
print("   PART B: STANDALONE LAYER 2 4-CLASS RANDOM FOREST (ESI 2, 3, 4, 5 on Non-ESI 1 Cohort)")
print("=" * 105)
print(report_l2_rf.to_string(index=False))
print("=" * 105 + chr(10))

# ---------------------------------------------------------
# Part C: RAW 2-STAGE HIERARCHICAL PROBABILITY CHAIN (WITHOUT LOGISTIC REGRESSOR)
# ---------------------------------------------------------
preds_raw_chain = np.argmax(test_probs, axis=1) + 1
report_raw_chain = get_per_class_breakdown(y_test, preds_raw_chain, test_probs, 'Raw 2-Stage Chain (Without LogReg)')

print("=" * 95)
print("   PART C: RAW 2-STAGE PROBABILITY CHAIN (WITHOUT LOGISTIC REGRESSOR)")
print("=" * 95)
print(report_raw_chain.to_string(index=False))
print("=" * 95 + chr(10))

# ---------------------------------------------------------
# Part D: FINAL CALIBRATED STACKING PIPELINE (WITH LOGISTIC REGRESSOR META-LEARNER)
# ---------------------------------------------------------
report_final = get_per_class_breakdown(y_test, preds_test, probs_test, 'Final 2-Stage Stacking Pipeline (With LogReg Meta on 20 Features)')

print("=" * 95)
print("   PART D: FINAL CALIBRATED STACKING PIPELINE (WITH LOGISTIC REGRESSOR META-LEARNER)")
print("=" * 95)
print(report_final.to_string(index=False))
print("=" * 95 + chr(10))

# Export reports to CSV
reports_dir = f'{ROOT}/reports'
os.makedirs(reports_dir, exist_ok=True)

l1_report_df.to_csv(os.path.join(reports_dir, 'oof_stacking_layer1_binary_report.csv'), index=False)
report_l2_rf.to_csv(os.path.join(reports_dir, 'oof_stacking_layer2_rf_report.csv'), index=False)
report_raw_chain.to_csv(os.path.join(reports_dir, 'oof_stacking_raw_chain_report.csv'), index=False)
report_final.to_csv(os.path.join(reports_dir, 'oof_stacking_final_pipeline_report.csv'), index=False)
print(f"✓ All reports successfully exported to {reports_dir}/")

In [ ]:
# ---------------------------------------------------------
# Step 7: Diagnostic Visualizations (5x5 Confusion Matrix & Multiclass ROC-AUC Curves)
# ---------------------------------------------------------
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

plots_dir = f"{ROOT}/plots"
os.makedirs(plots_dir, exist_ok=True)
os.makedirs(os.path.join(plots_dir, 'image'), exist_ok=True)

esi_labels = [f"ESI {i}" for i in range(1, 6)]

# 1. Left: 5x5 Normalized Confusion Matrix
cm      = confusion_matrix(y_test, preds_test, labels=[1, 2, 3, 4, 5])
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

annot = np.empty_like(cm, dtype=object)
for i in range(5):
    for j in range(5):
        annot[i, j] = f"{cm[i, j]:,}\n({cm_norm[i, j]*100:.1f}%)"

fig, axes = plt.subplots(1, 2, figsize=(18, 7.5))

sns.heatmap(
    cm_norm, annot=annot, fmt='', cmap='Blues', cbar=True, ax=axes[0],
    vmin=0, vmax=1, xticklabels=esi_labels, yticklabels=esi_labels
)
axes[0].set_title(
    f"Final 2-Stage Stacking (L1: LightGBM, L2: RF class_weight='balanced')\n"
    f"Macro Balanced Acc: {report_final.loc[5, 'Balanced_Accuracy']*100:.2f}% | Macro ROC-AUC: {report_final.loc[5, 'ROC_AUC']:.4f}",
    fontsize=11.5, fontweight='bold', pad=12
)
axes[0].set_xlabel("Predicted ESI Level", fontsize=11, fontweight='bold')
axes[0].set_ylabel("True ESI Level", fontsize=11, fontweight='bold')

# 2. Right: Multiclass ROC-AUC Curves
y_test_bin = label_binarize(y_test, classes=[1, 2, 3, 4, 5])
classes = [1, 2, 3, 4, 5]
n_classes = len(classes)
esi_colors = ['#d62728', '#ff7f0e', '#2ca02c', '#1f77b4', '#9467bd']

fpr, tpr, roc_aucs = dict(), dict(), dict()
for i, cls in enumerate(classes):
    fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], probs_test[:, i])
    roc_aucs[i] = auc(fpr[i], tpr[i])

fpr["micro"], tpr["micro"], _ = roc_curve(y_test_bin.ravel(), probs_test.ravel())
roc_aucs["micro"] = auc(fpr["micro"], tpr["micro"])

all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_classes)]))
mean_tpr = np.zeros_like(all_fpr)
for i in range(n_classes):
    mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
mean_tpr /= n_classes
fpr["macro"] = all_fpr
tpr["macro"] = mean_tpr
roc_aucs["macro"] = auc(fpr["macro"], tpr["macro"])

axes[1].plot(fpr["micro"], tpr["micro"], label=f"Micro-Average (AUC = {roc_aucs['micro']:.4f})", color='#e377c2', linestyle=':', linewidth=2.5)
axes[1].plot(fpr["macro"], tpr["macro"], label=f"Macro-Average (AUC = {roc_aucs['macro']:.4f})", color='#17becf', linestyle='--', linewidth=2.5)

for i in range(5):
    axes[1].plot(fpr[i], tpr[i], color=esi_colors[i], linewidth=2.0, label=f"ESI {i+1} (AUC = {roc_aucs[i]:.4f})")

axes[1].plot([0, 1], [0, 1], 'k--', color='gray', linewidth=1.2, label='Random Guess (AUC = 0.5000)')
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel('False Positive Rate (1 - Specificity)', fontsize=11, fontweight='bold')
axes[1].set_ylabel('True Positive Rate (Recall / Sensitivity)', fontsize=11, fontweight='bold')
axes[1].set_title(f"Final 2-Stage Stacking Multiclass ROC Curves (Macro AUC = {report_final.loc[5, 'ROC_AUC']:.4f})", fontsize=12, fontweight='bold', pad=10)
axes[1].legend(loc="lower right", fontsize=9.5, frameon=True, framealpha=0.95)
axes[1].grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
cm_plot_path = os.path.join(plots_dir, "oof_stacking_final_pipeline_evaluation.png")
plt.savefig(cm_plot_path, dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(plots_dir, 'image', 'oof_stacking_final_pipeline_evaluation.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Final Pipeline Diagnostic Plots saved to: {cm_plot_path}")

In [ ]:
# ---------------------------------------------------------
# Step 8: Export Production Artifact Bundle & Metadata Manifest
# ---------------------------------------------------------
deploy_dir = f'{ROOT}/deploy'
os.makedirs(deploy_dir, exist_ok=True)

bundle = {
    'scaler': scaler,
    'l1_lightgbm': l1_model,
    'l2_random_forest': l2_rf_model,
    'meta_learner': meta_learner,
    'feature_names': feature_names_20,
    'n_features': 20
}

bundle_file = os.path.join(deploy_dir, 'oof_stacking_final_pipeline_bundle.pkl')
with open(bundle_file, 'wb') as f:
    pickle.dump(bundle, f)

manifest = dict(
    pipeline='OOF_Stacking_Final_2Stage_LightGBM_RF_ClassWeight_LogRegMeta_20Feats',
    dataset='datasets/5v_cleandf.RData',
    n_classes=5,
    classes=['ESI 1', 'ESI 2', 'ESI 3', 'ESI 4', 'ESI 5'],
    n_features=20,
    feature_names=feature_names_20,
    holdout_test_samples=len(y_test),
    layer1_binary_metrics=l1_report_df.to_dict(orient='records'),
    layer2_rf_metrics=report_l2_rf.to_dict(orient='records'),
    raw_chain_metrics=report_raw_chain.to_dict(orient='records'),
    final_pipeline_metrics=report_final.to_dict(orient='records')
)

manifest_file = os.path.join(deploy_dir, 'oof_stacking_final_pipeline_manifest.json')
with open(manifest_file, 'w') as f:
    json.dump(manifest, f, indent=2)

print(f"✓ Final Pipeline Bundle   : {bundle_file}")
print(f"✓ Final Pipeline Manifest : {manifest_file}")